In [3]:
!pip install -q openai pinecone pypdf tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 16.0 MB/s eta 0:00:00


In [4]:
import os
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
from pypdf import PdfReader
from google.colab import files

# API Keys
os.environ["OPENAI_API_KEY"] = "sk-proj"
os.environ["PINECONE_API_KEY"] = ""

client = OpenAI()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [11]:
index_name = "rag-demo"

# Ensure the index is created with the correct dimension
if index_name in pc.list_indexes().names():
    pc.delete_index(index_name)

pc.create_index(
    name=index_name,
    dimension=1536,  # for text-embedding-3-small
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

index = pc.Index(index_name)

In [6]:
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

Saving Session2_LLMs_APIs_Workbook-20260417113201.pdf to Session2_LLMs_APIs_Workbook-20260417113201.pdf


In [7]:
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + "\n"
    return text

def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

In [8]:
def get_embedding(text):
    return client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    ).data[0].embedding

In [12]:
text = load_pdf(pdf_path)
chunks = chunk_text(text)

vectors = []
for i, chunk in enumerate(chunks):
    emb = get_embedding(chunk)
    vectors.append({
        "id": str(i),
        "values": emb,
        "metadata": {"text": chunk}
    })

index.upsert(vectors=vectors)
print("Stored in Pinecone ✅")

Stored in Pinecone ✅


In [13]:
def retrieve(query, top_k=3):
    query_emb = get_embedding(query)

    results = index.query(
        vector=query_emb,
        top_k=top_k,
        include_metadata=True
    )

    docs = [match["metadata"]["text"] for match in results["matches"]]
    return docs

In [14]:
def stream_answer(query, docs):
    context = "\n\n".join(docs)

    prompt = f"""
Answer ONLY from the given context.

Context:
{context}

Question:
{query}
"""

    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        stream=True
    )

    full_text = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            token = chunk.choices[0].delta.content
            print(token, end="", flush=True)
            full_text += token

    print()
    return full_text

In [15]:
while True:
    q = input("Ask (type 'exit' to stop): ")
    if q.lower() == "exit":
        break

    docs = retrieve(q)
    print("Answer:")
    stream_answer(q, docs)

Ask (type 'exit' to stop): Tell me overview
Answer:
The context provided is from a training session on Large Language Models (LLMs) and their applications, specifically focusing on building summarization and translation agents. It includes multiple-choice questions about key characteristics of LLMs, the concept of a context window, handling API rate limits, and considerations for choosing between different models for summarization tasks. The session emphasizes the importance of understanding token limits, output consistency, and the specific needs of tasks like summarization and translation.
Ask (type 'exit' to stop): exit
